# 🐭 El ratón, el queso y el laberinto

### Aprendizaje por refuerzo explicado como un juego

> **Para quién es esto:** para cualquiera. No hace falta saber matemáticas. Si entiendes
> "el ratón busca el queso", entiendes el aprendizaje por refuerzo.

> 📖 **¿Quieres entender el código, no solo la historia?** Este notebook tiene una guía
> hermana que explica **cada línea**: qué hace, por qué está escrita así y qué habría pasado
> escribiéndola de otra forma → **`03-el-raton-y-el-laberinto-EXPLICADO.md`**, en esta misma
> carpeta. Ábrela al lado; sus apartados van numerados igual que las celdas de código de aquí.

---

## El cuento

Metemos un ratón en un laberinto. En una esquina hay queso.

**Nadie le dice al ratón por dónde ir.** No hay mapa, no hay flechas en el suelo, no hay
un profesor diciéndole "gira a la derecha". Lo único que pasa es esto:

* Si da un paso y **no** encuentra queso → *"bah, has gastado energía"* 😐
* Si da un paso y **encuentra el queso** → *"¡QUESO!"* 🧀🎉

Y con eso, y solo con eso, el ratón acaba aprendiendo el camino.

---

## ¿Por qué esto es distinto de aprender "lo normal"?

Imagina que quieres que un niño aprenda a reconocer perros. Le enseñas mil fotos y en cada
una le dices **"esto es un perro"**. El niño ve la respuesta correcta cada vez. Eso es
**aprendizaje supervisado**: alguien te da las respuestas.

Ahora imagina que quieres que ese niño aprenda a montar en bici. **No puedes decirle la
respuesta**, porque no hay una respuesta que escribir: hay que probar, caerse, y notar qué
sienta bien. Nadie le dice "inclina el manillar 3 grados"; solo se cae o no se cae.

**Eso es el aprendizaje por refuerzo.** Y es el ratón de este notebook.

| | Aprender qué es un perro | Aprender a montar en bici |
|---|---|---|
| ¿Alguien te da la respuesta? | **Sí**, en cada foto | **No**, nunca |
| ¿Qué recibes? | La etiqueta correcta | Solo *"bien"* o *"mal"* |
| ¿Hay que probar cosas? | No | **Sí, es la única forma** |
| Cómo se llama | Supervisado | **Refuerzo** |

---

## El recorrido

```none
1. El laberinto            ── lo dibujamos y lo miramos
2. Las reglas del juego    ── qué puede hacer el ratón y qué gana
3. El ratón recién nacido  ── da vueltas a lo tonto (y eso es IMPRESCINDIBLE)
4. A entrenar              ── 15 segundos de práctica
5. La chuleta              ── ⭐ lo que ha aprendido, dibujado con flechas
6. Movemos el queso        ── la sorpresa: ¿de verdad aprendió a buscar queso?
7. Aparece un gato         ── aprende a dar un rodeo
8. Dos caminos igual buenos── no hay UNA respuesta correcta
9. El diccionario          ── traducimos el cuento a la jerga de verdad
```

## 0. Preparar las cosas

Usamos `lab/harness_rl.py`, el arnés de refuerzo del laboratorio. Es pequeño a propósito:
implementa el algoritmo más simple que funciona (se llama **REINFORCE**) y nada más.

In [ ]:
# ── ARRANQUE ──
import os, sys
from pathlib import Path

while not (Path.cwd() / "lab").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from lab import harness as H
from lab import harness_rl as rl

CPU = torch.device("cpu")
print("✅ Listo. Vamos a jugar.")

---
## 1. El laberinto

Cinco filas, cinco columnas. Veinticinco casillas.

* 🐭 el ratón empieza **arriba a la izquierda**
* 🧀 el queso está **abajo a la derecha**
* ⬛ los muros no se pueden cruzar

Fíjate en la forma: hay **dos pasillos** para bajar, uno por la izquierda y otro por la
derecha. Los dos son igual de largos. Guarda ese detalle, que en la sección 8 será
importante.

In [ ]:
MAPA = [
    "R....",
    ".###.",
    ".....",
    ".###.",
    "....Q",
]

FILAS, COLS = len(MAPA), len(MAPA[0])
N_CASILLAS  = FILAS * COLS

MUROS  = {(r, c) for r, fila in enumerate(MAPA)
                 for c, ch in enumerate(fila) if ch == "#"}
INICIO = (0, 0)
QUESO  = (4, 4)

# Las 4 cosas que puede hacer el ratón: (cuánto baja, cuánto se mueve al lado)
ACCIONES = {0: (-1, 0),    # arriba
            1: (+1, 0),    # abajo
            2: (0, -1),    # izquierda
            3: (0, +1)}    # derecha
FLECHAS  = {0: "↑", 1: "↓", 2: "←", 3: "→"}


def dibujar_texto(raton=INICIO, queso=QUESO, gato=None):
    '''Pinta el laberinto con emojis.'''
    lineas = []
    for r in range(FILAS):
        fila = []
        for c in range(COLS):
            if   (r, c) in MUROS:        fila.append("⬛")
            elif (r, c) == queso:        fila.append("🧀")
            elif gato and (r, c) == gato: fila.append("🐱")
            elif (r, c) == raton:        fila.append("🐭")
            else:                        fila.append("··")
        lineas.append(" ".join(fila))
    return "\n".join(lineas)


print(dibujar_texto())
print()
print(f"Casillas libres: {N_CASILLAS - len(MUROS)} de {N_CASILLAS}")

---
## 2. Las reglas del juego

### Lo que el ratón puede hacer

Cuatro cosas, ni una más: **arriba, abajo, izquierda, derecha**.

Si intenta ir contra un muro, se queda donde está (y ha perdido el turno).

### Lo que el ratón gana o pierde

| Lo que pasa | Qué recibe | En cristiano |
|---|---|---|
| Da un paso normal | **−0,05** | *"has gastado un poquito de energía"* |
| Llega al queso 🧀 | **+1,00** | *"¡PREMIO!"* — y se acaba la partida |
| Se choca con un gato 🐱 | **−1,00** | *"¡ay!"* — y se acaba la partida |

Ese **−0,05** de cada paso es pequeño pero importantísimo: es lo que hace que el ratón
prefiera el camino **corto**. Sin él, dar veinte vueltas antes de llegar al queso valdría
exactamente lo mismo que ir directo.

> 🧮 El camino más corto son **8 pasos**. Así que lo mejor que puede sacar el ratón es
> `1,00 − 8 × 0,05 = ` **+0,60**. Ese número es nuestra nota máxima: al final compararemos
> con él.

### Lo que el ratón ve

Y aquí está el detalle importante, porque es más limitado de lo que parece:

**el ratón solo sabe en qué casilla está.** Nada más. No ve el laberinto, no ve dónde está
el queso, no sabe si hay un muro delante hasta que se choca. Es como estar a oscuras con
un GPS que solo dice "estás en la casilla 7".

Eso se representa con 25 números: veinticuatro ceros y **un uno** en su casilla.

In [ ]:
def donde_estoy(posicion):
    '''Los 25 números que ve el ratón: todo ceros menos su casilla.'''
    v = torch.zeros(N_CASILLAS)
    v[posicion[0] * COLS + posicion[1]] = 1.0
    return v


print("Si el ratón está en la esquina de salida (fila 0, columna 0), esto es lo que ve:")
print(donde_estoy((0, 0)).numpy().astype(int))
print()
print("Y si estuviera justo al lado del queso (fila 4, columna 3):")
print(donde_estoy((4, 3)).numpy().astype(int))
print()
print("→ Solo cambia DÓNDE está el uno. El ratón no tiene ninguna otra información.")

Ahora escribimos las reglas en código. Esto es lo que en la jerga se llama un **entorno**:
la parte del programa que hace de "mundo" y que responde a lo que hace el ratón.

Fíjate en que el entorno se define **aquí, en el notebook**, no dentro de `harness_rl.py`.
Es a propósito: la librería trae el motor, y cada problema trae su propio mundo.

In [ ]:
@rl.envs.register("laberinto")
def construir_laberinto(gato=None, queso=QUESO, coste_paso=0.05, **kwargs):
    '''El mundo del ratón. Tres métodos: empezar, dar un paso, y ya está.'''

    class Laberinto:
        obs_dim   = N_CASILLAS   # cuántos números ve el ratón
        n_actions = 4            # cuántas cosas puede hacer

        def reset(self):
            '''Empieza una partida nueva: el ratón vuelve a la salida.'''
            self.pos = INICIO
            self.camino = [INICIO]      # guardamos por dónde ha pasado, para dibujarlo
            return donde_estoy(self.pos)

        def step(self, accion):
            '''El ratón intenta moverse. Devolvemos: dónde acaba, qué gana, si terminó.'''
            df, dc = ACCIONES[accion]
            destino = (self.pos[0] + df, self.pos[1] + dc)

            dentro = 0 <= destino[0] < FILAS and 0 <= destino[1] < COLS
            if dentro and destino not in MUROS:
                self.pos = destino          # se mueve
            # si no, se queda donde estaba: se ha chocado y ha perdido el turno

            self.camino.append(self.pos)

            if self.pos == queso:
                return donde_estoy(self.pos), 1.0 - coste_paso, True    # ¡PREMIO!
            if gato and self.pos == gato:
                return donde_estoy(self.pos), -1.0, True                # ¡ay!
            return donde_estoy(self.pos), -coste_paso, False             # paso normal

    return Laberinto()


print("✅ Reglas del laberinto registradas en el arnés")
print("   Entornos disponibles ahora:", sorted(rl.envs))

---
## 3. El ratón recién nacido

Creamos un ratón que **no sabe nada**. Su "cerebro" es una red neuronal con los pesos
puestos al azar, así que sus decisiones son básicamente monedas al aire.

Vamos a soltarlo y ver qué hace.

In [ ]:
H.set_seed(0)
raton_novato = H.models.build("mlp_policy", obs_dim=N_CASILLAS,
                              n_actions=4, hidden=64)

env = rl.envs.build("laberinto")
rl.collect_episode(raton_novato, env, CPU, max_steps=40)

print(f"El ratón dio {len(env.camino) - 1} pasos.")
print(f"Acabó en la casilla {env.pos}. El queso está en {QUESO}.")
print(f"¿Encontró el queso? {'SÍ 🧀' if env.pos == QUESO else 'NO 😐'}")
print()
print("Por dónde anduvo (en orden):")
print("  " + " → ".join(str(p) for p in env.camino[:14]) + " → ...")
print()
repetidas = sum(1 for a, b in zip(env.camino, env.camino[1:]) if a == b)
print(f"Veces que se quedó en el sitio (se chocó con un muro): {repetidas}")
print("Cuando ves la misma casilla dos veces seguidas, es el ratón dándose")
print("de morros contra una pared. No sabe que está ahí hasta que se choca.")

Da vueltas sin sentido, como es de esperar. Pero ahora viene **la pregunta más importante
de todo el notebook**:

> Si el ratón se mueve al azar... **¿cómo va a aprender nunca nada?**

La respuesta es que a veces, **por pura casualidad**, el paseo aleatorio acaba en el queso.
Y de esas casualidades sale absolutamente todo. Vamos a contarlas:

In [ ]:
exitos = 0
INTENTOS = 200
for _ in range(INTENTOS):
    rl.collect_episode(raton_novato, env, CPU, max_steps=40)
    if env.pos == QUESO:
        exitos += 1

print(f"De {INTENTOS} paseos completamente a lo tonto,")
print(f"encontraron el queso por casualidad: {exitos}  ({100*exitos/INTENTOS:.0f}%)")
print()
print("Ahí está TODO el secreto. Esos pocos paseos afortunados son las únicas veces")
print("que alguien le dice al ratón '¡bien!'. El aprendizaje consiste en mirar esos")
print("paseos y hacer que se repitan más a menudo.")

> 🔑 **La idea, en una frase:** el ratón no aprende el camino. Aprende a **repetir más lo
> que le salió bien y menos lo que le salió mal**. El camino aparece solo, como
> consecuencia.

Y por eso el porcentaje de arriba es tan crítico. Si fuera **0%** —si el ratón nunca
tropezara con el queso ni por suerte— **no habría nada que reforzar y no aprendería jamás**.

Ese es uno de los problemas de verdad del aprendizaje por refuerzo, y tiene nombre:
**el problema de la exploración**. En un laberinto de 5×5 se resuelve solo. En un
videojuego donde hay que hacer 300 cosas seguidas antes de ver el primer premio, es la
razón por la que las cosas no funcionan.

---
## 4. A entrenar

Ahora dejamos que practique. El arnés va a repetir este ciclo 120 veces:

```none
   ┌──────────────────────────────────────────────────────────┐
   │  1. El ratón juega 48 partidas con lo que sabe ahora     │
   │  2. Mira cuáles salieron bien y cuáles mal               │
   │  3. Ajusta un poquito su cerebro:                        │
   │        lo que salió bien  →  hazlo MÁS                   │
   │        lo que salió mal   →  hazlo MENOS                 │
   └──────────────────────────────────────────────────────────┘
                          ...y otra vez, 120 veces
```

Tarda unos **10 segundos**. Los números que van saliendo son la nota media de las 48
partidas de cada ronda: empieza mal y debería subir hasta acercarse a **+0,60**.

In [ ]:
config = {
    "name": "raton_laberinto",
    "env": "laberinto",
    "model": "mlp_policy",
    "model_args": {"hidden": 64},
    "optimizer": "adam",
    "optimizer_args": {"lr": 0.05},

    "iterations": 120,               # cuántas rondas de práctica
    "episodes_per_iteration": 48,    # cuántas partidas por ronda
    "max_steps": 40,                 # si no llega en 40 pasos, se acaba la partida

    "gamma": 0.99,                   # cuánta paciencia tiene (1 = mucha)
    "advantage": "normalized",
    "entropy_coef": 0.01,            # le pagamos un poco por seguir probando cosas
    "grad_clip": 1.0,
    "eval_episodes": 10,
    "seed": 0,
}

resultado = rl.run_rl_experiment(config)
print(f"\n💾 Guardado en runs/{resultado.run_id}/")

In [ ]:
historia = H.load_run(resultado.run_id)["history"]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(historia["epoch"], historia["reward_mean"], lw=2, color="steelblue",
        label="nota media del ratón")
ax.axhline(0.60, color="seagreen", ls="--", lw=2, label="lo máximo posible (+0,60)")
ax.axhline(0.0, color="grey", ls=":", alpha=0.6)
ax.set_xlabel("rondas de práctica")
ax.set_ylabel("nota (recompensa)")
ax.set_title("El ratón aprendiendo")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Nota al empezar : {historia['reward_mean'].iloc[0]:+.3f}")
print(f"Nota al terminar: {historia['reward_mean'].iloc[-1]:+.3f}")
print(f"Nota máxima posible: +0.600")
print()
print(f"Indecisión al empezar : {historia['entropy'].iloc[0]:.3f}")
print(f"Indecisión al terminar: {historia['entropy'].iloc[-1]:.3f}")
print(f"(el máximo con 4 opciones es ln(4) = {np.log(4):.3f}: no tener NINGUNA preferencia)")

---
## 5. ⭐ La chuleta: lo que ha aprendido el ratón

Esta es la parte bonita.

El "cerebro" del ratón no es más que **una opinión sobre qué hacer en cada casilla**. Y
como el laberinto solo tiene 25 casillas, podemos preguntárselo **una por una** y dibujar
su respuesta como una flecha.

Es literalmente **abrirle la cabeza y leer lo que piensa**.

In [ ]:
def chuleta(raton, gato=None, queso=QUESO):
    '''Pregunta al ratón qué haría en cada casilla y lo dibuja con flechas.'''
    lineas = []
    for r in range(FILAS):
        fila = []
        for c in range(COLS):
            if   (r, c) in MUROS:         fila.append("⬛")
            elif (r, c) == queso:         fila.append("🧀")
            elif gato and (r, c) == gato:  fila.append("🐱")
            else:
                with torch.no_grad():
                    mejor = int(raton(donde_estoy((r, c))).argmax())
                fila.append(f" {FLECHAS[mejor]}")
        lineas.append("  ".join(fila))
    return "\n".join(lineas)


print("Lo que el ratón haría en cada casilla del laberinto:\n")
print(chuleta(resultado.model))

Lo que ha aprendido **no es una lista de instrucciones** ("primero derecha, luego derecha,
luego abajo..."), sino **una opinión para cada sitio en el que pueda estar**. En la jerga
esto se llama una **política**, y es el objeto que el aprendizaje por refuerzo construye.

Y ahora una pregunta que parece tonta y no lo es: **¿funcionarían esas flechas si soltáramos
al ratón en una casilla cualquiera**, en vez de en la salida? Vamos a comprobarlo casilla por
casilla: seguimos su flecha, y luego la siguiente, y vemos si acabamos en el queso.

In [ ]:
def seguir_flechas(desde, raton, max_pasos=40, queso=QUESO):
    '''Suelta al ratón en una casilla y le hace seguir sus propias flechas.'''
    pos = desde
    for _ in range(max_pasos):
        if pos == queso:
            return True
        with torch.no_grad():
            a = int(raton(donde_estoy(pos)).argmax())
        df, dc = ACCIONES[a]
        destino = (pos[0] + df, pos[1] + dc)
        if 0 <= destino[0] < FILAS and 0 <= destino[1] < COLS and destino not in MUROS:
            pos = destino
    return False


# La ruta que recorre de verdad el ratón ENTRENADO (no la del novato de antes)
env_ruta = rl.envs.build("laberinto")
rl.collect_episode(resultado.model, env_ruta, CPU, max_steps=40, greedy=True)
su_ruta = set(env_ruta.camino)

libres = [(r, c) for r in range(FILAS) for c in range(COLS) if (r, c) not in MUROS]
llegan = [p for p in libres if seguir_flechas(p, resultado.model)]
fallan = [p for p in libres if p not in llegan]

print(f"Casillas libres del laberinto            : {len(libres)}")
print(f"Desde las que las flechas llevan al queso: {len(llegan)}")
print(f"Desde las que NO llevan a ningún sitio   : {len(fallan)}  →  {fallan}")
print()
for p in fallan:
    en_ruta = p in su_ruta
    print(f"  ¿El ratón pasa alguna vez por la casilla {p}? "
          f"{'sí' if en_ruta else 'NO, nunca pisa esa casilla'}")

Casi todas funcionan. Y las que fallan son justo las que **el ratón nunca visita**.

Búscalas en el dibujo de arriba: su flecha apunta **contra un muro**. Un ratón soltado ahí se
quedaría dándose de morros para siempre.

¿Por qué? Porque **esa flecha nunca se corrigió**. Sigue puesta donde la dejó el azar del
primer día. Para arreglarla haría falta que el ratón hubiese pasado por ahí, se hubiese
chocado y hubiese notado que era mala idea — y por ahí no pasó nunca.

> 🔑 **La lección, y es de las importantes:** este algoritmo solo mejora la política **donde
> el agente va**. No aprende nada de los sitios que no pisa. En la jerga se dice que es
> ***on-policy***: aprende de lo que hace, no de lo que podría hacer.
>
> Suena a detalle técnico y no lo es. Significa que si tu agente encuentra pronto una forma
> decente de resolver algo, **deja de mirar el resto del mapa** — y nunca sabrás si había
> algo mejor un poco más allá.

Vamos a verlo mejor dibujado, con el camino que recorre de verdad.

> 🎨 En los dibujos de matplotlib no usamos emojis (la fuente no los tiene y saldrían
> cuadraditos vacíos), así que la leyenda es: **círculo azul con R** = por donde sale el
> ratón · **estrella amarilla** = el queso · **X roja** = el gato · **línea roja** = el
> camino que recorre · **flechas grises** = su chuleta.

In [ ]:
def dibujar(raton=None, gato=None, queso=QUESO, camino=None, titulo="", ax=None):
    '''Dibuja el laberinto, las flechas de la política y el camino recorrido.

    Nota: aquí NO usamos emojis. matplotlib tira de la fuente DejaVu Sans, que no
    los tiene, y saldrían cuadraditos vacíos. Usamos marcadores de toda la vida:
    círculo azul = salida del ratón · estrella amarilla = queso · X roja = gato.
    '''
    solo = ax is None
    if solo:
        _, ax = plt.subplots(figsize=(5.4, 5.4))

    for r in range(FILAS):
        for c in range(COLS):
            if (r, c) in MUROS:
                ax.add_patch(Rectangle((c - .5, r - .5), 1, 1, color="#3a3a44"))
            elif raton is not None and (r, c) != queso and (r, c) != gato:
                with torch.no_grad():
                    a = int(raton(donde_estoy((r, c))).argmax())
                df, dc = ACCIONES[a]
                ax.arrow(c - dc * .22, r - df * .22, dc * .42, df * .42,
                         head_width=.16, head_length=.14,
                         fc="#8899aa", ec="#8899aa", lw=1.4)

    if camino:
        ys, xs = zip(*camino)
        ax.plot(xs, ys, color="crimson", lw=2.6, alpha=.85, zorder=3,
                marker="o", ms=4, label=f"camino ({len(camino)-1} pasos)")
        if solo:
            ax.legend(loc="upper center", bbox_to_anchor=(.5, -.03), fontsize=9)

    # SALIDA del ratón: círculo azul con una R
    ax.plot(INICIO[1], INICIO[0], marker="o", ms=17, mfc="#4a7fd0",
            mec="white", mew=1.6, zorder=4)
    ax.text(INICIO[1], INICIO[0], "R", ha="center", va="center", zorder=5,
            color="white", fontsize=9, fontweight="bold")
    # QUESO: estrella amarilla
    ax.plot(queso[1], queso[0], marker="*", ms=27, mfc="#f0b400",
            mec="#7a5c00", mew=1.2, zorder=4)
    # GATO: X roja
    if gato:
        ax.plot(gato[1], gato[0], marker="X", ms=19, mfc="#d0433f",
                mec="white", mew=1.6, zorder=4)

    ax.set_xticks(range(COLS)); ax.set_yticks(range(FILAS))
    ax.set_xlim(-.5, COLS - .5); ax.set_ylim(FILAS - .5, -.5)
    ax.set_aspect("equal"); ax.grid(color="#cccccc", lw=.8)
    ax.set_title(titulo, fontsize=11)
    return ax


env = rl.envs.build("laberinto")
rl.collect_episode(resultado.model, env, CPU, max_steps=40, greedy=True)

dibujar(resultado.model, camino=env.camino,
        titulo="Las flechas que aprendió, y el camino que sigue")
plt.tight_layout(); plt.show()

print(f"Pasos que da: {len(env.camino) - 1}   (el mínimo posible es 8)")
print(f"¿Llega al queso? {'SÍ 🧀' if env.pos == QUESO else 'NO'}")

---
## 6. 🎭 La sorpresa: movemos el queso

Aquí es donde el cuento se pone interesante, y donde aprendemos algo que mucha gente que
trabaja con esto se olvida.

El ratón **resuelve el laberinto perfectamente**. Ocho pasos, el mínimo. Parece listísimo.

Ahora hacemos una travesura: **cogemos el mismo ratón entrenado y movemos el queso a otra
esquina.** No lo volvemos a entrenar, no le decimos nada. Solo movemos el queso.

¿Qué crees que hará?

In [ ]:
QUESO_NUEVO = (4, 0)      # antes estaba en (4, 4)

env_travesura = rl.envs.build("laberinto", queso=QUESO_NUEVO)
rl.collect_episode(resultado.model, env_travesura, CPU, max_steps=40, greedy=True)

print(f"Pasos dados : {len(env_travesura.camino) - 1}  (se agotó el tiempo en 40)")
print(f"Acabó en    : {env_travesura.pos}")
print(f"El queso está en: {QUESO_NUEVO}")
print(f"¿Lo encontró? {'SÍ' if env_travesura.pos == QUESO_NUEVO else 'NO 😐'}")
print()
print("Sus últimas 8 posiciones:")
print("  " + " → ".join(str(p) for p in env_travesura.camino[-8:]))

In [ ]:
dibujar(resultado.model, queso=QUESO_NUEVO, camino=env_travesura.camino,
        titulo="Mismo ratón, queso movido: se va a la esquina de siempre")
plt.tight_layout(); plt.show()

### El ratón se va a la esquina vacía y se queda ahí, chocándose contra la pared.

Cuarenta pasos empujando una esquina donde ya no hay nada.

Y esto es importantísimo, porque revela **qué había aprendido en realidad**:

> ❌ El ratón **no** aprendió *"busca queso"*.
> ✅ El ratón aprendió *"camina hasta la esquina de abajo a la derecha"*.

Nunca supo que existía el queso. Nunca lo vio. Solo notó que cuando llegaba a cierta
casilla pasaba algo bueno, y se aprendió el camino hasta esa casilla de memoria.

**Ha memorizado, no ha entendido.**

> 🎓 Esto tiene nombre y es uno de los problemas centrales del campo: se llama **falta de
> generalización**. El agente resuelve exactamente el problema que practicó y se rompe con
> el cambio más pequeño.
>
> ¿Cómo se arregla? **Entrenando en muchos laberintos distintos** en vez de en uno solo, y
> dejando que el ratón **vea** dónde está el queso en lugar de solo su propia casilla. Es
> la diferencia entre aprenderse un examen de memoria y aprender la asignatura.

Y de paso explica algo que sale en las noticias: cuando lees que una IA "juega al ajedrez
mejor que nadie" pero no sabe jugar a las damas, es este mismo ratón.

---
## 7. 🐱 Aparece un gato

Volvemos a poner el queso en su sitio, pero ahora metemos un **gato** en el pasillo de la
derecha, en la casilla (2, 4).

El gato no se mueve. Simplemente está ahí, y si el ratón lo pisa: **−1,00** y partida
terminada.

Acuérdate de la sección 1: **había dos pasillos para bajar**. El ratón entrenado antes
usaba el de la derecha... que es justo el que acaba de quedar cortado.

Lo entrenamos de nuevo, desde cero, con el gato puesto.

In [ ]:
GATO = (2, 4)

print("El laberinto ahora:\n")
print(dibujar_texto(gato=GATO))

In [ ]:
config_gato = dict(config, name="raton_con_gato", env_args={"gato": GATO})
resultado_gato = rl.run_rl_experiment(config_gato, verbose=False)

env_gato = rl.envs.build("laberinto", gato=GATO)
rl.collect_episode(resultado_gato.model, env_gato, CPU, max_steps=40, greedy=True)

print(f"Pasos: {len(env_gato.camino) - 1}")
print(f"¿Llegó al queso? {'SÍ 🧀' if env_gato.pos == QUESO else 'NO'}")
print(f"¿Pasó por el gato? {'SÍ 😿' if GATO in env_gato.camino else 'NO, lo esquivó ✓'}")

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(11, 5.4))

env_sin = rl.envs.build("laberinto")
rl.collect_episode(resultado.model, env_sin, CPU, max_steps=40, greedy=True)
dibujar(resultado.model, camino=env_sin.camino,
        titulo="Sin gato: baja por la derecha", ax=izq)

dibujar(resultado_gato.model, gato=GATO, camino=env_gato.camino,
        titulo="Con gato: las flechas se dan la vuelta", ax=der)

plt.tight_layout(); plt.show()

### Compara los dos dibujos: **todas las flechas se han dado la vuelta.**

El ratón ha aprendido a bajar por el pasillo de la izquierda. Y lo bonito es que **nadie
le dijo que había un gato**. Nadie le explicó "cuidado, peligro a la derecha". Simplemente:

1. Al principio probaba el pasillo derecho, como cualquiera.
2. Se encontraba el gato y se llevaba un **−1,00**.
3. Ese castigo hacía **bajar** la probabilidad de ir por ahí.
4. Repetido unas cuantas veces, el pasillo derecho dejó de ser una opción.

En este laberinto el rodeo **sale gratis**: los dos pasillos miden lo mismo, así que el
ratón sigue llegando en 8 pasos con nota +0,60. Esquivar el peligro no le ha costado nada.

> ⚠️ **Un aviso honesto.** Si el gato bloqueara el **único** camino y el rodeo fuera mucho
> más largo, este ratón **no lo encontraría**. Lo he comprobado: en una versión del
> laberinto donde el desvío obliga a dar 12 pasos en vez de 8, **0 de 3 intentos lo
> consiguen** — el ratón se pasa la partida entera dando vueltas sin volver a ver el queso.
>
> Es el problema de la exploración de la sección 3, otra vez, y en serio. Arreglarlo
> necesita cosas que este arnés mínimo **no tiene** a propósito (una segunda red que estime
> "lo bueno" de cada casilla, y algoritmos como PPO). Están en el mapa 🔀 al final de
> `lab/harness_rl.py`.

---
## 8. Dos caminos igual de buenos

Última cosa, y es bonita.

Los dos pasillos miden lo mismo. Así que **no hay una respuesta correcta**: hay dos, y son
igual de buenas.

¿Qué hace el ratón entonces? Pues **depende de la suerte**. De cuál de los dos encontró
primero por casualidad. Entrenamos cinco ratones idénticos, cambiando solo el número de la
suerte inicial (la *semilla*):

In [ ]:
ratones = {}
for semilla in range(5):
    res = rl.run_rl_experiment(dict(config, seed=semilla, name=f"raton_s{semilla}"),
                               verbose=False, save=False)
    env_s = rl.envs.build("laberinto")
    rl.collect_episode(res.model, env_s, CPU, max_steps=40, greedy=True)
    ratones[semilla] = (res, env_s.camino, env_s.pos)

print("  ratón │ pasos │ llega │ ¿por qué pasillo bajó?")
print("  ──────┼───────┼───────┼───────────────────────")
for semilla, (res, camino, fin) in ratones.items():
    columnas_medias = [p[1] for p in camino if p[0] == 2]     # por dónde cruzó la fila 2
    lado = "izquierda" if columnas_medias and min(columnas_medias) == 0 else "derecha"
    print(f"    {semilla}   │   {len(camino)-1:2d}  │  {'SÍ' if fin == QUESO else 'NO':2s}   │ {lado}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(19, 4.2))
for ax, (semilla, (res, camino, fin)) in zip(axes, ratones.items()):
    dibujar(res.model, camino=camino, titulo=f"ratón {semilla}", ax=ax)
    ax.set_xticklabels([]); ax.set_yticklabels([])
plt.tight_layout(); plt.show()

Cinco ratones, **el mismo laberinto**, el mismo entrenamiento, la misma nota final. Y unos
bajan por la izquierda y otros por la derecha.

> 💡 **Óptimo no significa único.** Cuando hay varias soluciones igual de buenas, lo que
> decide es el azar de los primeros tanteos. No hay nada que "descubrir": hay que **elegir**,
> y la elección se congela en cuanto el ratón deja de explorar.

Esto también es un aviso metodológico serio: **si entrenas un agente UNA vez, no sabes qué
has medido.** En este laberinto todos los ratones sacan la misma nota, así que da igual.
Pero en la sección 7 viste que con el gato uno de cada cuatro se atasca del todo — y si
hubieras entrenado justo ése, tu conclusión habría sido "esto no funciona".

---
## 9. 📖 El diccionario

Todo lo que has hecho en este notebook tiene nombres técnicos. Aquí está la traducción, y
ya está: si has entendido la columna de la izquierda, entiendes la de la derecha.

| En el cuento del ratón | Cómo se llama de verdad | Dónde está en el código |
|---|---|---|
| El ratón (su forma de decidir) | La **política** ($\pi$) | `mlp_policy` |
| En qué casilla está | El **estado** ($s$) | `donde_estoy()` |
| Dar un paso | La **acción** ($a$) | `ACCIONES` |
| El laberinto y sus reglas | El **entorno** | `@rl.envs.register("laberinto")` |
| Lo que gana o pierde | La **recompensa** ($R$) | lo que devuelve `step()` |
| Una partida completa | Un **episodio** / trayectoria | `rl.collect_episode()` |
| Cuánta paciencia tiene | **gamma** ($\gamma$) | `"gamma": 0.99` |
| La chuleta de flechas | La política aprendida | `chuleta()` |
| Probar caminos nuevos | **Exploración** | `entropy_coef` |
| Ir por el que ya conoce | **Explotación** | evaluación `greedy=True` |
| "Haz más de lo que salió bien" | **Gradiente de política** (REINFORCE) | `rl.policy_loss()` |

---

# 💡 Lo que hay que llevarse

### Si te quedas con tres cosas

1. **Nadie le dijo al ratón el camino.** Solo "bien" o "mal", y de ahí salió todo. Eso es
   el aprendizaje por refuerzo, y es como aprendemos a montar en bici.
2. **Hace falta suerte al principio.** El 14% de paseos que encontraban el queso por
   casualidad **son** el material de aprendizaje. Sin esas casualidades no hay nada que
   reforzar. Y por eso los problemas donde el premio está lejísimo son tan difíciles.
3. **Aprender no es entender.** El ratón resolvía el laberinto perfecto y se estampaba
   contra una esquina vacía en cuanto movimos el queso. Memorizó una ruta, no aprendió a
   buscar.

### Y una cuarta, para quien vaya a usar esto de verdad

4. **Una sola ejecución no te dice nada.** Cinco ratones idénticos eligieron caminos
   distintos, y con el gato uno de cada cuatro fracasó por completo. Entrena varias veces
   siempre (`rl.repeat_rl_with_seeds`).

---

### Dónde seguir

| Si quieres... | Ve a |
|---|---|
| Ver el **mecanismo matemático** (por qué esto funciona) | `notebooks/02-refuerzo-con-el-arnes-rl.ipynb` |
| Entender **por qué RL no cabe** en el arnés normal | `lab/HARNESS.md` §4 |
| Saber **qué le falta** a este arnés y qué usar en su lugar | `lab/harness_rl.py`, sección 🔀 |
| Ver cómo esto **alinea modelos de lenguaje** (RLHF) | `notebooks/01-aprendizaje-multi-etapa-arnes.ipynb` |

> 🐭 Y si te ha gustado: cambia el `MAPA` de la sección 1 y vuelve a ejecutar todo. Mueve
> los muros, pon dos gatos, haz el laberinto más grande. Todo lo demás funciona igual —
> es lo bonito de tener las reglas separadas del algoritmo.